# Phân tích khám phá dữ liệu (EDA) thị trường cho thuê RoomBeacon

Notebook này bao gồm các phần sau:

- [Nhập các thư viện](#nhập-các-thư-viện)

- [Kết nối DuckDB và tải tập dữ liệu](#kết-nối-duckdb-và-tải-tập-dữ-liệu)

- [Tổng quan về dataset](#tổng-quan-về-dataset)

- [Đánh giá chất lượng dữ liệu](#đánh-giá-chất-lượng-dữ-liệu)

- [Phân tích nguồn dữ liệu](#phân-tích-nguồn-dữ-liệu)

- [Phân tích giá thuê](#phân-tích-giá-thuê)

- [Phân tích diện tích cho thuê](#phân-tích-diện-tích-cho-thuê)

- [Phát hiện giá trị ngoại lệ](#phát-hiện-giá-trị-ngoại-lệ)

- [Phân tích mối quan hệ](#phân-tích-mối-quan-hệ)

- [Phân tích theo thời gian](#phân-tích-theo-thời-gian)

- [Tóm tắt nhận định dữ liệu](#tóm-tắt-nhận-định-dữ-liệu)

## Nhập các thư viện
Phần này nhập các thư viện Python cần thiết để xử lý dữ liệu, trực quan hóa và thực hiện phân tích khám phá dữ liệu.

In [78]:
try:
    from utils import setup_project_path
except ModuleNotFoundError:
    from notebooks.utils import setup_project_path

PROJECT_ROOT = setup_project_path()

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=False)
from utils.location_normalizer import normalize_location
import numpy as np
import pandas as pd
from analytics.duckdb.connection import create_analytics_connection

print("Đã nhập thư viện và thiết lập môi trường thành công.")

Đã phát hiện thư mục gốc của dự án RoomBeacon: /mnt/Data/projects/roombeacon
Đã nhập thư viện và thiết lập môi trường thành công.


## Kết nối DuckDB và tải tập dữ liệu

Phần này kết nối với cơ sở dữ liệu DuckDB của RoomBeacon và tải tập dữ liệu phân tích vào Pandas DataFrame để thực hiện phân tích khám phá.

In [50]:
# Factory của dự án tạo catalog DuckDB, gắn Bronze MySQL bằng cú pháp DSN
# key=value tương thích với DuckDB và khởi tạo các analytical view.
conn = create_analytics_connection()

attached_databases = {row[0] for row in conn.execute("SHOW DATABASES").fetchall()}
if "mysql_db" not in attached_databases:
    raise RuntimeError(
        "DuckDB không thể gắn Bronze MySQL. Hãy bảo đảm mysql-bronze đang chạy "
        "và thông tin kết nối BRONZE_MYSQL_HOST/PORT trong .env có thể truy cập "
        "từ môi trường notebook hiện tại."
    )

In [51]:
print("Đã kết nối thành công!")
conn.sql("SHOW TABLES;").show()

Đã kết nối thành công!
┌──────────────────────────┐
│           name           │
│         varchar          │
├──────────────────────────┤
│ v_acquisition_efficiency │
│ v_content_changes        │
│ v_data_quality           │
│ v_latest_posts           │
│ v_listing_lifetime       │
│ v_location_summary       │
│ v_observations           │
│ v_price_history          │
│ v_source_activity        │
└──────────────────────────┘



In [52]:
df = conn.sql("SELECT * FROM v_latest_posts").df()
type(df)

pandas.DataFrame

## Tổng quan về dataset

Phần này cung cấp cái nhìn tổng quan về tập dữ liệu được sử dụng trong quá trình EDA.

Các mục tiêu chính bao gồm:

- Kiểm tra kích thước tổng thể của tập dữ liệu.
- Xác nhận cấu trúc và lược đồ của dữ liệu.
- Kiểm tra thông tin các trường dữ liệu.
- Quan sát một số bản ghi mẫu.
- Đánh giá sơ bộ các biến số quan trọng trước khi thực hiện phân tích chuyên sâu.

Các nội dung thực hiện:

- Kích thước tập dữ liệu:
    - Số lượng bản ghi.
    - Số lượng đặc trưng.

- Xem trước tập dữ liệu:
    - Quan sát dữ liệu mẫu.
    - Kiểm tra định dạng và giá trị ban đầu.

- Thông tin tập dữ liệu:
    - Tên các cột.
    - Kiểu dữ liệu.
    - Số lượng giá trị không bị thiếu.

- Thống kê cơ bản:
    - Thống kê mô tả các biến số.
    - Kiểm tra phạm vi giá trị của các trường dạng số.

Kết quả của phần này giúp xác định cấu trúc tập dữ liệu và chuẩn bị cho các bước tiếp theo như đánh giá chất lượng dữ liệu và phân tích khám phá.

### Kích thước tập dữ liệu
Kiểm tra kích thước tổng thể của tập dữ liệu bao gồm:

- Số lượng bản ghi (hàng).
- Số lượng đặc trưng (cột).

In [53]:
df.shape

(80654, 14)

In [54]:
df.columns

Index(['source_code', 'rental_post_id', 'source_listing_id', 'title_raw',
       'url', 'price_amount', 'area_value', 'full_address_text',
       'location_raw', 'full_address_inherited', 'latest_observed_at',
       'first_observed_at', 'last_observed_at', 'active_days'],
      dtype='str')

In [55]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 80654 entries, 0 to 80653
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   source_code             80654 non-null  str           
 1   rental_post_id          80654 non-null  int64         
 2   source_listing_id       80654 non-null  str           
 3   title_raw               80190 non-null  str           
 4   url                     80654 non-null  str           
 5   price_amount            71811 non-null  float64       
 6   area_value              69406 non-null  float64       
 7   full_address_text       17937 non-null  str           
 8   location_raw            17937 non-null  str           
 9   full_address_inherited  80654 non-null  bool          
 10  latest_observed_at      80654 non-null  datetime64[us]
 11  first_observed_at       80654 non-null  datetime64[us]
 12  last_observed_at        80654 non-null  datetime64[us]
 1

### Thông tin các trường dữ liệu

Tập dữ liệu `v_latest_posts` gồm 14 trường dữ liệu chính, đại diện cho thông tin định danh, thuộc tính, vị trí và vòng đời của tin đăng cho thuê.

Các nhóm trường chính:

### Thông tin định danh và nguồn dữ liệu

- `source_code`: Nguồn dữ liệu được thu thập, xác định nền tảng đăng tin.
- `rental_post_id`: Mã định danh nội bộ duy nhất của RoomBeacon cho mỗi tin đăng.
- `source_listing_id`: Mã định danh gốc của tin đăng trên trang web nguồn.
- `url`: Đường dẫn truy cập tin đăng gốc.

### Thuộc tính tin đăng

- `title_raw`: Tiêu đề nguyên bản của tin đăng.
- `price_amount`: Giá thuê được bóc tách dạng số (VNĐ/tháng).
- `area_value`: Diện tích tin đăng (m²).

### Thông tin vị trí

- `full_address_text`: Địa chỉ chi tiết thu thập từ nguồn.
- `location_raw`: Trường địa chỉ thô phục vụ phân tích vị trí.
- `full_address_inherited`: Đánh dấu địa chỉ được lấy trực tiếp hay kế thừa từ lịch sử quan sát.

### Thời gian và vòng đời

- `latest_observed_at`: Thời điểm hệ thống thu thập quan sát tin gần nhất.
- `first_observed_at`: Thời điểm đầu tiên phát hiện tin đăng.
- `last_observed_at`: Thời điểm cuối cùng ghi nhận hoạt động.
- `active_days`: Số ngày tin đăng tồn tại trên hệ thống.

Các trường dữ liệu này phục vụ cho các bước phân tích tiếp theo như:
- Đánh giá chất lượng dữ liệu.
- Phân tích giá thuê và diện tích.
- Phân tích nguồn dữ liệu.
- Nghiên cứu vòng đời tin đăng.

### Xem trước tập dữ liệu
- Quan sát dữ liệu mẫu.
- Kiểm tra định dạng và giá trị ban đầu.


In [56]:
df.head(5)

,source_code,rental_post_id,source_listing_id,title_raw,url,price_amount,area_value,full_address_text,location_raw,full_address_inherited,latest_observed_at,first_observed_at,last_observed_at,active_days
0,mogi,790893,22653642,Cho Thuê Phòng Trọ Giá Rẻ Sinh Viên ngay Ngã t...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,4000000.0,26.0,"Bạch Đằng, Phường 24, Quận Bình Thạnh, TPHCM","Bạch Đằng, Phường 24, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:52:06,2026-08-28 03:51:22,2026-08-30 16:52:06,2
1,mogi,790892,22653655,Cho Thuê Phòng trọ giá rẻ cửa sổ ngay tại Ngã ...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,4000000.0,30.0,"Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM","Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:52:04,2026-08-28 03:51:20,2026-08-30 16:52:04,2
2,mogi,790891,22653658,Cho Thuê Phòng trọ sinh viên zá rẻ ngay Ngã Tư...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,3000000.0,25.0,"Nguyễn Gia Trí, Phường 25, Quận Bình Thạnh, TPHCM","Nguyễn Gia Trí, Phường 25, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:52:02,2026-08-28 03:51:20,2026-08-30 16:52:02,2
3,mogi,790890,22655584,Cho Thuê Phòng Trọ Giá Rẻ Cho Sinh Viên Ngay N...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,5000000.0,35.0,"Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM","Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:52:00,2026-08-28 03:51:20,2026-08-30 16:52:00,2
4,mogi,790889,22653695,Cho Thuê Phòng trọ giá rẻ ngay Ngã Tư Hàng Xan...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,4000000.0,30.0,"Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM","Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:51:58,2026-08-28 03:51:20,2026-08-30 16:51:58,2


In [57]:
# Kiểm tra số bản ghi có full_address_inherited = true
address_inherited = conn.query("SELECT COUNT(*) FROM v_latest_posts WHERE full_address_inherited = 1")
print(address_inherited)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1620 │
└──────────────┘



In [58]:
# Tổng quan về các đặc trưng dạng số
# Khảo sát phân phối thống kê của các trường dạng số quan trọng:
# price_amount, area_value và active_days.

overview_features = [
    "rental_post_id",
    "price_amount",
    "area_value",
    "active_days",
    "full_address_text",
    "location_raw"
]

df[overview_features].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
rental_post_id,80654.0,NaN,NaN,NaN,823423.581248,39839.540251,757140.0,788999.25,821677.5,857443.75,898676.0
price_amount,71811.0,NaN,NaN,NaN,5021010.501817,38432390.110018,3500.0,3000000.0,4000000.0,6000000.0,7777000000.0
area_value,69406.0,NaN,NaN,NaN,31.789789,27.415207,1.0,25.0,30.0,35.0,2300.0
active_days,80654.0,NaN,NaN,NaN,0.396608,0.921418,0.0,0.0,0.0,0.0,6.0
full_address_text,17937,6675,"Căn 02.34, Lầu 2, Tháp 3, The Sun Avenue, Số 2...",3124,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_raw,17937,6675,"Căn 02.34, Lầu 2, Tháp 3, The Sun Avenue, Số 2...",3124,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Tổng quan các đặc trưng của tập dữ liệu

Thống kê mô tả cung cấp góc nhìn ban đầu về số lượng giá trị, phân phối và phạm vi của các trường quan trọng. Những quan sát này là cơ sở để xác định nội dung cần kiểm tra sâu hơn, chưa phải kết luận cuối cùng về chất lượng dữ liệu.

## Đánh giá chất lượng dữ liệu

Phần này đánh giá chất lượng dữ liệu trước khi thực hiện làm sạch, tập trung vào dữ liệu thiếu, dữ liệu trùng lặp và các vấn đề về tính hợp lệ. Các phát hiện sẽ được sử dụng để xây dựng quy tắc làm sạch phù hợp cho tập dữ liệu RoomBeacon.

### Phân tích dữ liệu thiếu

**Mục tiêu:**

- Kiểm tra mức độ thiếu dữ liệu của các trường quan trọng.
- Xác định các trường dữ liệu cần được điều tra thêm trước khi xây dựng quy tắc làm sạch.

Dữ liệu thiếu chưa đồng nghĩa với lỗi dữ liệu. Với `price_amount`, giá trị `NULL` có thể xuất hiện khi trang web không cung cấp giá số, tin đăng hiển thị "Thỏa thuận", bộ phân tích cú pháp chưa trích xuất được dữ liệu hoặc các nguồn sử dụng định dạng khác nhau. Cần kiểm tra bằng chứng từ dữ liệu gốc trước khi kết luận nguyên nhân.

### Điều tra dữ liệu thiếu giá

Biểu thức `missing_price = df[df["price_amount"].isna()]` lọc các tin đăng chưa có giá số để phục vụ điều tra. Việc lấy thêm `source_code`, `url` và `title_raw` giúp:

- Truy ngược nguồn dữ liệu.
- Đối chiếu nội dung thực tế trên trang web.
- Xác định nguyên nhân thiếu dữ liệu dựa trên bằng chứng thay vì suy đoán.

In [59]:
# Kiểm tra các giá trị NaN
missing_price = df[df["price_amount"].isna()]
missing_price[
    [
        "source_code",
        "url",
        "title_raw"
    ]
].head()

,source_code,url,title_raw
302,tromoi,https://tromoi.com/can-ho/can-ho-cao-cap-canh-...,Căn hộ cao cấp cạnh ngay vivo city quận 7
332,cafeland,https://nhadat.cafeland.vn/moi-gioi/trong-van-...,Trọng Văn
336,cafeland,https://nhadat.cafeland.vn/phong-tro-pass-tro-...,Phòng trọ: Pass trọ quận 7
339,cafeland,https://nhadat.cafeland.vn/phong-tro-phong-dep...,"Phòng trọ: Phòng Đẹp, Máy Lạnh, Gần Ngã Tư 4 X..."
351,cafeland,https://nhadat.cafeland.vn/phong-tro-phong-20m...,Phòng trọ: Phòng 20m² có gác sạch đẹp – Khu 5E...


In [60]:
df["source_code"].value_counts()

source_code
chothuephongtro    35640
mogi               29700
cafeland           10009
nhatot              2174
phongtro123         1153
tromoi              1019
nhatrovn             883
chothuenha            76
Name: count, dtype: int64

# Đánh giá chất lượng dữ liệu

Phần này đánh giá chất lượng dữ liệu hiện tại trước khi thực hiện các bước làm sạch và chuẩn hóa.

Mục tiêu của bước này là xác định các vấn đề có thể ảnh hưởng đến quá trình phân tích và xây dựng các quy tắc xử lý dữ liệu phù hợp.

Các kiểm tra chính bao gồm:

- **Dữ liệu thiếu**: Đánh giá mức độ thiếu dữ liệu của các trường quan trọng như giá thuê, diện tích và thông tin vị trí.
- **Kiểm tra tính hợp lệ của dữ liệu**: Kiểm tra kiểu dữ liệu, khoảng giá trị và các ràng buộc nghiệp vụ.
- **Kiểm tra trùng lặp**: Phát hiện các bản ghi trùng lặp trong tập dữ liệu.
- **Kiểm tra thông tin vị trí**: Kiểm tra và chuẩn hóa thông tin địa chỉ theo cấu trúc hành chính.

Các vấn đề phát hiện trong bước này sẽ được sử dụng để xây dựng quy trình làm sạch và chuyển đổi trước khi tạo tập dữ liệu phục vụ cho các bước phân tích tiếp theo.

In [61]:
df.dtypes

source_code                          str
rental_post_id                     int64
source_listing_id                    str
title_raw                            str
url                                  str
price_amount                     float64
area_value                       float64
full_address_text                    str
location_raw                         str
full_address_inherited              bool
latest_observed_at        datetime64[us]
first_observed_at         datetime64[us]
last_observed_at          datetime64[us]
active_days                        int64
dtype: object

In [62]:
df.isnull().sum()

source_code                   0
rental_post_id                0
source_listing_id             0
title_raw                   464
url                           0
price_amount               8843
area_value                11248
full_address_text         62717
location_raw              62717
full_address_inherited        0
latest_observed_at            0
first_observed_at             0
last_observed_at              0
active_days                   0
dtype: int64

In [63]:
missing_rate = (
    df.isnull()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

missing_rate

location_raw              77.76
full_address_text         77.76
area_value                13.95
price_amount              10.96
title_raw                  0.58
rental_post_id             0.00
url                        0.00
source_listing_id          0.00
source_code                0.00
full_address_inherited     0.00
latest_observed_at         0.00
first_observed_at          0.00
last_observed_at           0.00
active_days                0.00
dtype: float64

In [64]:
df[df["full_address_text"].isna()]["source_code"].value_counts()

source_code
chothuephongtro    31473
mogi               25615
cafeland            5621
nhatot                 7
tromoi                 1
Name: count, dtype: int64

#### thay đổi tên các cột 

In [65]:
df_work = df.copy()

In [66]:
df_work.head(10)

,source_code,rental_post_id,source_listing_id,title_raw,url,price_amount,area_value,full_address_text,location_raw,full_address_inherited,latest_observed_at,first_observed_at,last_observed_at,active_days
0,mogi,790893,22653642,Cho Thuê Phòng Trọ Giá Rẻ Sinh Viên ngay Ngã t...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,4000000.0,26.0,"Bạch Đằng, Phường 24, Quận Bình Thạnh, TPHCM","Bạch Đằng, Phường 24, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:52:06,2026-08-28 03:51:22,2026-08-30 16:52:06,2
1,mogi,790892,22653655,Cho Thuê Phòng trọ giá rẻ cửa sổ ngay tại Ngã ...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,4000000.0,30.0,"Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM","Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:52:04,2026-08-28 03:51:20,2026-08-30 16:52:04,2
2,mogi,790891,22653658,Cho Thuê Phòng trọ sinh viên zá rẻ ngay Ngã Tư...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,3000000.0,25.0,"Nguyễn Gia Trí, Phường 25, Quận Bình Thạnh, TPHCM","Nguyễn Gia Trí, Phường 25, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:52:02,2026-08-28 03:51:20,2026-08-30 16:52:02,2
3,mogi,790890,22655584,Cho Thuê Phòng Trọ Giá Rẻ Cho Sinh Viên Ngay N...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,5000000.0,35.0,"Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM","Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:52:00,2026-08-28 03:51:20,2026-08-30 16:52:00,2
4,mogi,790889,22653695,Cho Thuê Phòng trọ giá rẻ ngay Ngã Tư Hàng Xan...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,4000000.0,30.0,"Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM","Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:51:58,2026-08-28 03:51:20,2026-08-30 16:51:58,2
5,mogi,790888,22654008,Cho Thuê Phòng Trọ Giá rẻ ngay Học Viện Cán Bộ...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,4000000.0,27.0,"Bùi Đình Túy, Phường 26, Quận Bình Thạnh, TPHCM","Bùi Đình Túy, Phường 26, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:51:56,2026-08-28 03:51:20,2026-08-30 16:51:56,2
6,mogi,790887,22655586,Cho Thuê phòng trọ Giá rẻ Ngay Đại Học Văn Lang,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,4000000.0,40.0,"Đặng Thùy Trâm, Phường 13, Quận Bình Thạnh, TPHCM","Đặng Thùy Trâm, Phường 13, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:51:54,2026-08-28 03:51:20,2026-08-30 16:51:54,2
7,mogi,790886,22654035,Cho Thuê Phòng trọ giá rẻ Siêu đẹp ngay Phố ẩm...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,4000000.0,27.0,"Phan Đăng Lưu, Phường 14, Quận Bình Thạnh, TPHCM","Phan Đăng Lưu, Phường 14, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:51:52,2026-08-28 03:51:20,2026-08-30 16:51:52,2
8,mogi,790885,22654891,Cho Thuê Phòng Trọ Giá Rẻ ngay Ngã Tư Hàng Xan...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,4000000.0,28.0,"Xô Viết Nghệ Tĩnh, Phường 25, Quận Bình Thạnh,...","Xô Viết Nghệ Tĩnh, Phường 25, Quận Bình Thạnh,...",False,2026-08-30 16:51:51,2026-08-28 03:51:20,2026-08-30 16:51:51,2
9,mogi,790884,22653406,Cho Thuê Căn Hộ Giá Rẻ_Ngay Ngã tư hàng xanh t...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,5000000.0,30.0,"Xô Viết Nghệ Tĩnh, Phường 25, Quận Bình Thạnh,...","Xô Viết Nghệ Tĩnh, Phường 25, Quận Bình Thạnh,...",False,2026-08-30 16:51:48,2026-08-28 03:51:20,2026-08-30 16:51:48,2


In [67]:
# rename columns 
df_work = df_work.rename(
    columns={
        "source_code": "domain",
        "rental_post_id": "roombeacon_id",
        "source_listing_id": "website_id",
        "title_raw": "title",
        "url": "post_url",
        "price_amount": "monthly_price",
        "area_value": "area_m2",
        "full_address_text": "address",
        "location_raw": "location",
        "full_address_inherited": "address_inherited",
        "latest_observed_at": "latest_seen_at",
        "first_observed_at": "first_seen_at",
        "last_observed_at": "last_seen_at",
        "active_days": "active_days"
    }
)

In [68]:
df_work.columns

Index(['domain', 'roombeacon_id', 'website_id', 'title', 'post_url',
       'monthly_price', 'area_m2', 'address', 'location', 'address_inherited',
       'latest_seen_at', 'first_seen_at', 'last_seen_at', 'active_days'],
      dtype='str')

In [69]:
df_work.head(5)

,domain,roombeacon_id,website_id,title,post_url,monthly_price,area_m2,address,location,address_inherited,latest_seen_at,first_seen_at,last_seen_at,active_days
0,mogi,790893,22653642,Cho Thuê Phòng Trọ Giá Rẻ Sinh Viên ngay Ngã t...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,4000000.0,26.0,"Bạch Đằng, Phường 24, Quận Bình Thạnh, TPHCM","Bạch Đằng, Phường 24, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:52:06,2026-08-28 03:51:22,2026-08-30 16:52:06,2
1,mogi,790892,22653655,Cho Thuê Phòng trọ giá rẻ cửa sổ ngay tại Ngã ...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,4000000.0,30.0,"Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM","Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:52:04,2026-08-28 03:51:20,2026-08-30 16:52:04,2
2,mogi,790891,22653658,Cho Thuê Phòng trọ sinh viên zá rẻ ngay Ngã Tư...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,3000000.0,25.0,"Nguyễn Gia Trí, Phường 25, Quận Bình Thạnh, TPHCM","Nguyễn Gia Trí, Phường 25, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:52:02,2026-08-28 03:51:20,2026-08-30 16:52:02,2
3,mogi,790890,22655584,Cho Thuê Phòng Trọ Giá Rẻ Cho Sinh Viên Ngay N...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,5000000.0,35.0,"Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM","Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:52:00,2026-08-28 03:51:20,2026-08-30 16:52:00,2
4,mogi,790889,22653695,Cho Thuê Phòng trọ giá rẻ ngay Ngã Tư Hàng Xan...,https://mogi.vn/quan-binh-thanh/thue-phong-tro...,4000000.0,30.0,"Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM","Đinh Bộ Lĩnh, Phường 26, Quận Bình Thạnh, TPHCM",False,2026-08-30 16:51:58,2026-08-28 03:51:20,2026-08-30 16:51:58,2


# histogram

In [83]:
from utils.location_normalizer import normalize_location

df_work = normalize_location(df_work)

TypeError: expected string or bytes-like object, got 'float'